# 5. Pre-determined risk set and a firm-year / technology-year design

**Why this notebook exists.** Notebook 4's null is a power result, not an economic one (the levels form is the exception — small, negative and significant on the movers sample, but sensitive to the variance estimator and running against both other forms, so treated there as mechanical rather than a result.)
Under firm × subclass FE the coefficient rests on 293 pairs across 151 firms, and the
interval on `net_any` covers both zero and an effect twice the descriptive cross-tab
(13.0% vs 8.6%).

The cause: 62% of the grid is cells the firm would never patent in. Pair FE remove those
structural zeros by absorbing them, along with nearly all usable variation. This notebook
removes them *ex ante* instead, which frees the fixed effects for other work.

**The design.** Risk set: keep firm × subclass combinations where the firm patented in that
subclass **before 2015**. Determined outside the estimation window, so no in-window outcome
is conditioned on — unlike filtering on whether `patented` varies, which selected on $y$ and
inflated the notebook 4 coefficient from 0.005 to 0.030.

Fixed effects:

$$y_{ist} = \beta\,x_{ist} + \alpha_{it} + \gamma_{st} + \varepsilon_{ist}$$

$\alpha_{it}$ (firm × year) absorbs everything happening to the firm that year — size,
financial conditions, group-wide innovation drives. $\gamma_{st}$ (subclass × year) absorbs
technology trends, so a subclass that is simply hot in 2021 cannot drive the result.

Identifying variation is within a firm-year, across technologies the firm already works in:
do those where the Italian network holds a stock see more new families than those where it
does not, net of each technology's own trend? Pair FE absorbed exactly this contrast.
`firm_year` and `sub_year` are already on the panel, so only the risk-set filter is new.

**What it costs.** The estimand narrows to the intensive margin — whether exposure raises
continued patenting in technologies the firm already works in. Entry into genuinely new
technologies is what the risk set removes, and stays in notebook 4 as the under-powered
result it is. Firm × year FE also absorb any firm-level control, so `financials.dta`
(open item 4) drops out by construction and the 2015–2024 window is no longer bound by
its coverage.

**What stays unidentified.** Firm × year FE do not absorb reallocation *within* the
firm-year: a group pushing C07D across all affiliates moves exposure and own-patenting
together, and subclass × year FE catch this only where the push is economy-wide. That
residual is the case for the instrument in notebook 7.

**First checks.**

1. How far back pre-2015 priority dates run in `patents.dta`.
2. How many of the 674 sample firms have a pre-2015 patent, and how many pairs the risk set yields.
3. Firm-years with **two or more** risk-set subclasses — with one, the cell contributes
   nothing once $\alpha_{it}$ is absorbed.
4. Share of risk-set cells with `net_stock` > 0.

If check 3 collapses, the design fails before estimation and notebook 6's variants come early.

In [1]:
import numpy as np
import pandas as pd
import pyfixest as pf

data_dir = r"C:\Users\Popov\Documents\Studies\IMT_studies\Projects\Econ_2\Datasets"
panel = pd.read_parquet(data_dir + r"\panel_firm_subclass_year.parquet")

FIRM, PARENT, SUB, YEAR = "bvd_key", "parent_key", "subclass", "year"
Y, X = "patented", "net_stock"

print(f"{len(panel):,} cells | {panel[FIRM].nunique():,} firms | "
      f"{panel[SUB].nunique():,} subclasses | {panel[YEAR].min()}-{panel[YEAR].max()}")

94,010 cells | 674 firms | 527 subclasses | 2015-2024


### Check 1 — pre-2015 patent history

The risk set is built on patenting before the panel window, so it depends entirely on how
far back `patents.dta` runs. Orbis IP coverage thins going backwards, and if the history is
effectively five years deep, firms that patented sporadically will drop out on timing rather
than inactivity.

In [2]:
patents = pd.read_stata(data_dir + r"\patents.dta")
print(patents.shape)
print(patents.dtypes)
patents.head()

(629225, 10)
publication_number                         str
priority_date                   datetime64[ms]
grant_date                      datetime64[ms]
number_of_family_members               float64
family_identifier__simple              float64
applicant_s__bvd_id_number_s               str
ipc_code__main                             str
cpc_code__main                             str
backward_citations                         str
bvd_key                                    str
dtype: object


,publication_number,priority_date,grant_date,number_of_family_members,family_identifier__simple,applicant_s__bvd_id_number_s,ipc_code__main,cpc_code__main,backward_citations,bvd_key
0,AD432020EA,2019-11-11,NaT,1.0,1.137924e+09,IT02574090276,,,,IT02574090276
1,AD432020PB,2019-11-11,NaT,1.0,1.137925e+09,IT02574090276,,,,IT02574090276
2,AE200400150A,NaT,2022-07-08,1.0,1.147881e+09,IT03998870962,B01J27/045,,,IT03998870962
3,AE200700334A,2004-10-18,NaT,1.0,1.137926e+09,IT03998870962,,,,IT03998870962
4,AE200900338A,2006-10-16,NaT,1.0,1.137936e+09,IT03998870962,A23L2/54,,,IT03998870962


In [3]:
PFIRM, PFAM, PDATE = "bvd_key", "family_identifier__simple", "priority_date"

pyear = patents[PDATE].dt.year
pre = patents[pyear < 2015]

print(pyear.describe(), "\n")
print(pyear[pyear < 2015].value_counts().sort_index().tail(20), "\n")
print(f"pre-2015 families: {pre[PFAM].nunique():,}")
print(f"firms with any pre-2015 family: {pre[PFIRM].nunique():,}")

count    412812.000000
mean       2004.721287
std          13.591546
min        1900.000000
25%        1998.000000
50%        2007.000000
75%        2015.000000
max        2025.000000
Name: priority_date, dtype: float64 

priority_date
1995.0     6315
1996.0     6655
1997.0     7110
1998.0     7786
1999.0     8359
2000.0    10911
2001.0    11837
2002.0    12761
2003.0    11228
2004.0    12358
2005.0    11942
2006.0    12406
2007.0    10401
2008.0     9863
2009.0    10883
2010.0    11714
2011.0    12508
2012.0    13829
2013.0    13771
2014.0    13938
Name: count, dtype: int64 

pre-2015 families: 83,259
firms with any pre-2015 family: 8,385


In [4]:
print(f"patents with no family id: {patents[PFAM].isna().sum():,}")

patents with no family id: 39,484


Coverage runs steadily back to the late 1990s at 10–14k families a year, with no thinning
at the recent end, so the 2015 cutoff is not a timing artifact. Note 216k of 629k patents
have no priority date and 39k no family id — the risk set is built on rows where both are
present. The 8,385 firms here are the whole file; what matters is how many of the 674
sample firms have pre-2015 history, which is check 2.

### Check 2 — risk set against the estimation sample

The risk set keeps firm × subclass combinations the firm patented in before 2015. Two
numbers matter: how many of the 674 sample firms have any pre-2015 history, and how many
firm-subclass pairs survive. Firms with no pre-2015 patent drop out entirely — the first
real cost of the design.

The membership check confirms the subclass is derived the same way as the panel's
(`ipc_code__main`, first 4 characters). Near 1.0 means the risk-set join will be clean;
well below means notebook 3 built `subclass` from a different field and the rule needs
aligning before anything joins.

In [5]:
sample_firms = set(panel[FIRM].unique())

valid = patents.dropna(subset=[PDATE, PFAM])
pre = valid[valid[PDATE].dt.year < 2015].copy()
pre["sub"] = pre["ipc_code__main"].str[:4]

rs = pre[pre[PFIRM].isin(sample_firms)][[PFIRM, "sub"]].drop_duplicates()

print(f"sample firms: {len(sample_firms):,}")
print(f"  ...with any pre-2015 patent: {rs[PFIRM].nunique():,}")
print(f"risk-set firm-subclass pairs: {len(rs):,}")
print(f"  subclasses: {rs['sub'].nunique():,}")

sample firms: 674
  ...with any pre-2015 patent: 516
risk-set firm-subclass pairs: 3,523
  subclasses: 550


In [6]:
print(f"risk-set subclasses in panel: {rs['sub'].isin(panel[SUB].unique()).mean():.3f}")

risk-set subclasses in panel: 0.940


In [7]:
missing = rs.loc[~rs["sub"].isin(panel[SUB].unique()), "sub"]
print(f"{missing.nunique()} subclasses, {len(missing)} pairs not in panel")
print(missing.value_counts().head(10))
print("\nlengths:", pre["sub"].str.len().value_counts().to_dict())

47 subclasses, 212 pairs not in panel
sub
        148
B21K      6
B82Y      4
F23J      3
F21Q      2
F15D      2
D06Q      2
G12B      2
B68G      2
F21M      2
Name: count, dtype: int64

lengths: {4: 288840, 0: 9400}


The join lands at 0.94, not 1.0. The gap is two harmless things: ~9,400 patents have a
blank `ipc_code__main` (Orbis missing data, written as `""` in notebook 1) and truncate to
an empty subclass; and ~46 real IPC subclasses exist pre-2015 but fall outside the panel's
window-restricted universe — technologies a firm worked in before 2015 and has since left.
Neither belongs in the risk set, so it is intersected with the panel's subclasses. That
intersection is the design's definition, not a patch: the risk set must live inside the
estimable universe.

In [8]:
rs = rs[(rs["sub"] != "") & rs["sub"].isin(panel[SUB].unique())]
print(f"risk-set pairs: {len(rs):,} | firms: {rs[PFIRM].nunique():,} | subclasses: {rs['sub'].nunique():,}")

risk-set pairs: 3,311 | firms: 516 | subclasses: 503


These firm and pair counts, after the intersection, are the real check 2 — the first
measure of what the risk set costs against the 674-firm sample.

### Check 3 — does the design have power?

The decisive check. Under firm × year FE, a firm-year identifies the coefficient only if it
holds **two or more** risk-set subclasses — with one, the FE absorbs it entirely. Three
numbers to read:

- **Share of firm-years with ≥2 subclasses.** If most have only one, firm × year FE leave
  nothing and the design is no better than notebook 4. A healthy share means it is live.
- **Identifying cells**, against notebook 4's 16,760. In that range or better is a gain;
  a few thousand means the intensive margin is as thin as the extensive one.
- **Risk-set cells with net_stock > 0** (check 4, folded in). Identification needs exposure
  to *differ* across a firm-year's subclasses — if every cell is exposed, there is no
  within-firm-year contrast to use.

If the first collapses, the design fails here and notebook 6's variants come early.

In [9]:
# Panel restricted to risk-set cells: firm patented in that subclass before 2015
rs_pairs = set(zip(rs[PFIRM], rs["sub"]))
panel["in_rs"] = [(f, s) in rs_pairs for f, s in zip(panel[FIRM], panel[SUB])]
rspanel = panel[panel["in_rs"]].copy()

print(f"risk-set panel: {len(rspanel):,} cells | {rspanel[FIRM].nunique():,} firms")

# Subclasses per firm-year — firm-years with <2 drop out under firm×year FE
fy = rspanel.groupby([FIRM, YEAR])[SUB].nunique()
print(f"\nfirm-years total: {len(fy):,}")
print(f"  with >=2 subclasses: {(fy >= 2).sum():,} ({(fy >= 2).mean():.1%})")
print(f"  identifying cells (in those firm-years): "
      f"{rspanel.groupby([FIRM, YEAR]).filter(lambda g: g[SUB].nunique() >= 2).shape[0]:,}")

# Exposure variation inside the risk set (check 4, same cell — it's cheap)
print(f"\nrisk-set cells with net_stock>0: {(rspanel[X] > 0).mean():.1%}")

risk-set panel: 27,440 cells | 516 firms

firm-years total: 5,160
  with >=2 subclasses: 3,310 (64.1%)
  identifying cells (in those firm-years): 25,590

risk-set cells with net_stock>0: 24.5%


This clears the bar. 64.1% of firm-years hold ≥2 subclasses, so the design survives
firm × year FE. The 25,590 identifying cells beat notebook 4's 16,760, and come from 516
firms rather than 151 — real added variation, not the same thin sample relabelled.

`net_stock > 0` in only 24.5% of cells is a feature, not a worry: it means exposed and
unexposed subclasses sit side by side within a firm-year, which is exactly the contrast
that identifies β. The skew toward zero does mean `net_any` and `log_net` will carry most
of the signal.

Note the risk set and the regressor filter on different patent sets: the risk set on the
firm's *own* pre-2015 patenting in the subclass, `net_stock` on the *network's* families
(co-affiliates, firm excluded). A risk-set cell can therefore have `net_stock = 0` — the
firm works in the technology, its siblings do not. That mismatch is the identifying
contrast, not a flaw.

Within a firm-year, across the subclasses the firm already works in, some have network presence and most don’t — and we are asking whether the ones that do see more new families in that subclass that year.

So the clean phrasing of the test: within a firm-year, across the subclasses the firm already works in, are the ones where the network holds a stock more likely to see a new family than the ones where it doesn’t — net of each subclass’s own trend that year.

## Estimation

Two FE structures. Headline is firm × year + subclass × year: identification is the
cross-subclass contrast within a firm-year, using the full risk-set variation. The
conservative version adds the firm-subclass pair FE, which also absorbs any persistent
firm-subclass level — if the coefficient survives it, the effect is not driven by a firm
simply patenting more in some of its established technologies than others.

In [10]:
rsp = rspanel.copy()
rsp["firm_id"] = rsp[FIRM].astype("category").cat.codes
rsp["pair"] = rsp[FIRM] + "|" + rsp[SUB]
cl = {"CRV1": "firm_id"}

print(f"{len(rsp):,} cells | {rsp['firm_id'].nunique():,} firms | {rsp['pair'].nunique():,} pairs")

27,440 cells | 516 firms | 2,744 pairs


### Headline — firm × year + subclass × year

`firm_year` drops singleton firm-years (one row), so N falls below 25,590 — those are the
single-subclass firm-years leaving, as expected. `net_any` and `log_net` carry the signal;
`net_stock` levels are noisier given 75% of cells are zero.

In [11]:
fe1 = "firm_year + sub_year"

a_stock = pf.feols(f"{Y} ~ {X}     | {fe1}", data=rsp, vcov=cl)
a_log   = pf.feols(f"{Y} ~ log_net | {fe1}", data=rsp, vcov=cl)
a_any   = pf.feols(f"{Y} ~ net_any | {fe1}", data=rsp, vcov=cl)

pf.etable([a_stock, a_log, a_any])

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


GT(_tbl_data=  __index_level_0__ __index_level_1__                   0  \
0              coef         net_stock  0.000 <br> (0.000)   
1              coef           log_net                       
2              coef           net_any                       
3                fe         firm_year                   -   
4                fe          sub_year                   -   
5             stats      Observations              24,320   
6             stats                R²               0.334   

                      1                      2  
0                                               
1  0.025** <br> (0.009)                         
2                        0.061*** <br> (0.015)  
3                     -                      -  
4                     -                      -  
5                24,320                 24,320  
6                 0.337                  0.339  , _body=<great_tables._gt_data.Body object at 0x00000259865FEF30>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x00000259865FFA40>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x00000259865FEFC0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x00000259865FF590>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x00000259865FF440>, _formats=[<great_tables._gt_data.FormatInfo object at 0x00000259865FEED0>, <great_tables._gt_data.FormatInfo object at 0x00000259865FF5F0>, <great_tables._gt_data.FormatInfo object at 0x00000259865FF1D0>, <great_tables._gt_data.FormatInfo object at 0x00000259865FF500>, <great_tables._gt_data.FormatInfo object at 0x00000259865FEBD0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color

### Conservative — add firm-subclass pair FE

Identification is now within-pair over time, with subclass trends netted out by `sub_year`.
Cleaner than notebook 4's within-pair contrast, but check the identifying count hasn't
collapsed back toward 293 — if it has, the pair FE has spent the variation the risk set
bought. Compare the coefficient to the headline: survives → not firm-subclass heterogeneity;
collapses → it was.

In [12]:
fe2 = "firm_year + sub_year + pair"

b_stock = pf.feols(f"{Y} ~ {X}     | {fe2}", data=rsp, vcov=cl)
b_log   = pf.feols(f"{Y} ~ log_net | {fe2}", data=rsp, vcov=cl)
b_any   = pf.feols(f"{Y} ~ net_any | {fe2}", data=rsp, vcov=cl)

pf.etable([b_stock, b_log, b_any])

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(


GT(_tbl_data=  __index_level_0__ __index_level_1__                    0  \
0              coef         net_stock  -0.000 <br> (0.000)   
1              coef           log_net                        
2              coef           net_any                        
3                fe         firm_year                    -   
4                fe              pair                    -   
5                fe          sub_year                    -   
6             stats      Observations               24,320   
7             stats                R²                0.598   

                     1                    2  
0                                            
1  -0.011 <br> (0.024)                       
2                       -0.014 <br> (0.028)  
3                    -                    -  
4                    -                    -  
5                    -                    -  
6               24,320               24,320  
7                0.598                0.598  , _body=<great_tables._gt_data.Body object at 0x00000259865BBC50>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x000002598B054EC0>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x000002598B054350>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x000002598B054560>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x000002598B054830>, _formats=[<great_tables._gt_data.FormatInfo object at 0x000002598656EBD0>, <great_tables._gt_data.FormatInfo object at 0x000002598B054230>, <great_tables._gt_data.FormatInfo object at 0x000002598B054D10>, <great_tables._gt_data.FormatInfo object at 0x000002598B054E60>, <great_tables._gt_data.FormatInfo object at 0x000002598B0544A0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_st

### What the two tables show

Read together, not separately.

The headline is large and significant: `net_any` 0.061 (0.015)\*\*\*, `log_net` 0.025
(0.009)\*\*. Against the risk-set outcome mean of **7.9%** — not the full panel's 3.6%, since
the risk set keeps only technologies the firm already patents in — network presence raises
the probability of continued patenting by about **77%**.

Adding the pair FE kills it: `net_any` −0.014 (0.028), `log_net` −0.011 (0.024). R² rises
from 0.34 to 0.60, so the pair FE absorb most of the variance — and the signal with it.

The headline effect is therefore **entirely cross-sectional**. It comes from comparing,
within a firm-year, subclasses where the network has a stock against subclasses where it
does not. Force identification to come from a given firm-subclass changing *over time* and the effect disappears. ("cross-sectional" here means the contrast is across technologies within a firm-year, not
that the data are a cross-section — the panel is the same in both specifications.)

That pattern is what sorting looks like, not spillover. A group operating in pharma has
both a network and an Italian affiliate patenting in pharma subclasses, because that is the
group's business — no knowledge needs to flow for the correlation to appear. The within-pair
test asks whether a *build-up* of network stock is followed by more affiliate patenting, and
finds no detectable response — with the power caveat below.

This is not a precise zero. The pair FE identify off 390 moving pairs (163 with the outcome
also moving) for the continuous forms, and 127 (56) for the binary — thinner than notebook
4's 3,069 and 293 — so the standard errors are wide: 0.028 on `net_any` against a risk-set outcome mean of 0.079.

What it does establish: the interval on the within-pair estimate, roughly [−0.069, +0.041],
**excludes the cross-sectional 0.061**. Sorting accounts for the bulk of the headline
effect. It does not establish that the spillover is zero — an effect up to +0.041 remains
inside the interval, and that is larger than the outcome mean.

| Margin | Result |
|---|---|
| Extensive (nb 4) | underpowered null |
| Intensive, cross-sectional | large, significant — confounded by sorting |
| Intensive, within-pair over time | rejects the cross-sectional magnitude; cannot rule out a moderate effect |

The gap between the second and third rows is the finding. It says the association between
network and affiliate patenting reflects shared technological identity, not knowledge flow —
consistent with notebook 2, where intra-group citation proved genuinely rare. Three
independent cuts point the same way.

**Example.** Firm A patented pre-2015 in C07D (pharma) and H04W (wireless); both are in its
risk set. In 2020 its sister firms hold 12 families in C07D and none in H04W.

Under `firm_year + sub_year`, the comparison is across the two subclasses within that
firm-year: is A more likely to file in C07D than in H04W? A's overall 2020 activity and each
subclass's 2020 trend are absorbed, but the comparison is still between two different
technologies — and A is simply a pharma firm.

Adding `pair` gives C07D and H04W their own intercepts, so that comparison disappears. What
remains is within C07D over time: the sisters held 3 families there in 2016 and 12 by 2022 —
does A's filing probability rise as the stock grows? Only the second comparison can be a
knowledge flow, and it is the one the data can least well measure.

In [13]:
# nice table - python
models = [a_stock, a_log, a_any, b_stock, b_log, b_any]

tab = pf.etable(models, type="df")
tab = tab.drop("fe", level=0)

# Own FE rows — only the last one varies across columns, which is the comparison
tab.loc[("fe", "Firm x year FE"), :]      = ["Yes"] * 6
tab.loc[("fe", "Subclass x year FE"), :]  = ["Yes"] * 6
tab.loc[("fe", "Firm-subclass FE"), :]    = ["No"] * 3 + ["Yes"] * 3

tab = tab.reindex(["coef", "fe", "stats"], level=0)
tab = tab.replace({r"\n": " "}, regex=True)   # the literal \n in the coef cells
tab

patented                     \
                                      (1)                (2)   
coef  net_stock           0.000   (0.000)                      
      log_net                              0.025**   (0.009)   
      net_any                                                  
fe    Firm x year FE                  Yes                Yes   
      Subclass x year FE              Yes                Yes   
      Firm-subclass FE                 No                 No   
stats Observations                 24,320             24,320   
      R²                            0.334              0.337   

                                                                \
                                         (3)               (4)   
coef  net_stock                               -0.000   (0.000)   
      log_net                                                    
      net_any             0.061***   (0.015)                     
fe    Firm x year FE                     Yes               Yes   
      Subclass x year FE                 Yes               Yes   
      Firm-subclass FE                    No               Yes   
stats Observations                    24,320            24,320   
      R²                               0.339             0.598   

                                                              
                                       (5)               (6)  
coef  net_stock                                               
      log_net             -0.011   (0.024)                    
      net_any                               -0.014   (0.028)  
fe    Firm x year FE                   Yes               Yes  
      Subclass x year FE               Yes               Yes  
      Firm-subclass FE                 Yes               Yes  
stats Observations                  24,320            24,320  
      R²                             0.598             0.598

In [14]:
out = tab.copy()
out.index = out.index.droplevel(0)
out = out.rename(index={"net_stock": "Network stock", "log_net": "log(1 + stock)",
                        "net_any": "Network present"})

print(out.to_latex(escape=False, column_format="l" + "c" * 6,
                   caption="Network exposure and continued patenting",
                   label="tab:risk_set"))

\begin{table}
\caption{Network exposure and continued patenting}
\label{tab:risk_set}
\begin{tabular}{lcccccc}
\toprule
 & \multicolumn{6}{r}{patented} \\
 & (1) & (2) & (3) & (4) & (5) & (6) \\
\midrule
Network stock & 0.000   (0.000) &  &  & -0.000   (0.000) &  &  \\
log(1 + stock) &  & 0.025**   (0.009) &  &  & -0.011   (0.024) &  \\
Network present &  &  & 0.061***   (0.015) &  &  & -0.014   (0.028) \\
Firm x year FE & Yes & Yes & Yes & Yes & Yes & Yes \\
Subclass x year FE & Yes & Yes & Yes & Yes & Yes & Yes \\
Firm-subclass FE & No & No & No & Yes & Yes & Yes \\
Observations & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 \\
R² & 0.334 & 0.337 & 0.339 & 0.598 & 0.598 & 0.598 \\
\bottomrule
\end{tabular}
\end{table}



### Identifying variation per regressor

The two FE structures use different variation, so each needs its own count. Under
`firm_year + sub_year`, a firm-year contributes only if the regressor differs *across* the
subclasses the firm works in that year. Adding the pair FE moves identification to variation
*over time* within a firm-subclass, which is the notebook 4 margin — and the pair counts
below should be read against notebook 4's 3,069 moving pairs and 293 with the outcome moving.

In [15]:
for v in [X, "log_net", "net_any"]:
    # fe1: regressor must vary ACROSS subclasses within a firm-year
    fy = rsp.groupby(["firm_id", YEAR]).agg(x=(v, "nunique"), y=(Y, "nunique"))
    fy_x = (fy.x > 1).sum()
    fy_both = ((fy.x > 1) & (fy.y > 1)).sum()

    # fe2: regressor must vary OVER TIME within a pair
    pr = rsp.groupby("pair").agg(x=(v, "nunique"), y=(Y, "nunique"))
    pr_x = (pr.x > 1).sum()
    pr_both = ((pr.x > 1) & (pr.y > 1)).sum()

    print(f"{v:10s} | firm-years: x varies {fy_x:5,} of {len(fy):,}, y also {fy_both:5,} "
          f"|| pairs: x moves {pr_x:5,} of {len(pr):,}, y also {pr_both:4,}")

net_stock  | firm-years: x varies 1,987 of 5,160, y also   579 || pairs: x moves   390 of 2,744, y also  163
log_net    | firm-years: x varies 1,987 of 5,160, y also   579 || pairs: x moves   390 of 2,744, y also  163
net_any    | firm-years: x varies 1,797 of 5,160, y also   562 || pairs: x moves   127 of 2,744, y also   56


### Foreign vs domestic parents

If knowledge flows from a multinational network at all, the foreign-parent subsample is
where it should appear — and this design can now see it. Run on both FE structures.

In [16]:
dom  = rsp[rsp["parent_is_foreign"] == 0].copy()
forn = rsp[rsp["parent_is_foreign"] == 1].copy()

for name, d in [("domestic", dom), ("foreign", forn)]:
    print(f"{name:9s} | {len(d):,} cells | {d['pair'].nunique():,} pairs | "
          f"{d['firm_id'].nunique():,} firms | {d[PARENT].nunique():,} groups")

FORMS = [X, "log_net", "net_any"]

s1, s2 = [], []
for v in FORMS:
    s1.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=dom,  vcov=cl))
    s1.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=forn, vcov=cl))
    s2.append(pf.feols(f"{Y} ~ {v} | {fe2}", data=dom,  vcov=cl))
    s2.append(pf.feols(f"{Y} ~ {v} | {fe2}", data=forn, vcov=cl))

pf.etable(s1)

domestic  | 19,450 cells | 1,945 pairs | 401 firms | 193 groups
foreign   | 7,990 cells | 799 pairs | 115 firms | 60 groups


c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2940 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1970 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2940 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 1970 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 2940 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\

GT(_tbl_data=  __index_level_0__ __index_level_1__                   0                   1  \
0              coef         net_stock  0.000 <br> (0.000)  0.000 <br> (0.000)   
1              coef           log_net                                           
2              coef           net_any                                           
3                fe         firm_year                   -                   -   
4                fe          sub_year                   -                   -   
5             stats      Observations              16,510               6,020   
6             stats                R²               0.382               0.458   

                    2                   3                      4  \
0                                                                  
1  0.018 <br> (0.010)  0.005 <br> (0.015)                          
2                                          0.055*** <br> (0.015)   
3                   -                   -                      -   
4                   -                   -                      -   
5              16,510               6,020                 16,510   
6               0.384               0.458                  0.386   

                    5  
0                      
1                      
2  0.046 <br> (0.053)  
3                   -  
4                   -  
5               6,020  
6                0.46  , _body=<great_tables._gt_data.Body object at 0x0000025986AA2300>, _boxhead=Boxhead([ColInfo(var='__index_level_0__', type=<ColInfoTypeEnum.row_group: 3>, column_label='__index_level_0__', column_align='center', column_width=None), ColInfo(var='__index_level_1__', type=<ColInfoTypeEnum.stub: 2>, column_label='__index_level_1__', column_align='center', column_width=None), ColInfo(var='0', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(1)'), column_align='center', column_width=None), ColInfo(var='1', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(2)'), column_align='center', column_width=None), ColInfo(var='2', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(3)'), column_align='center', column_width=None), ColInfo(var='3', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(4)'), column_align='center', column_width=None), ColInfo(var='4', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(5)'), column_align='center', column_width=None), ColInfo(var='5', type=<ColInfoTypeEnum.default: 1>, column_label=Html(text='(6)'), column_align='center', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x0000025986BA94F0>, _spanners=Spanners([SpannerInfo(spanner_id='patented', spanner_level=1, spanner_label=Html(text='patented'), spanner_units=None, spanner_pattern=None, vars=['0', '1', '2', '3', '4', '5'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x0000025986BA9310>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x0000025986BA9A00>, _source_notes=[Html(text='Significance levels: * p &lt; 0.05, ** p &lt; 0.01, *** p &lt; 0.001. Format of coefficient cell: Coefficient   (Std. Error)')], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x0000025986BAA240>, _formats=[<great_tables._gt_data.FormatInfo object at 0x0000025986BA9D60>, <great_tables._gt_data.FormatInfo object at 0x0000025986A56BD0>, <great_tables._gt_data.FormatInfo object at 0x0000025986BA9A60>, <great_tables._gt_data.FormatInfo object at 0x0000025986BA9400>, <great_tables._gt_data.FormatInfo object at 0x0000025986BA9B50>, <great_tables._gt_data.FormatInfo object at 0x0000025986BA9F10>, <great_tables._gt_data.FormatInfo object at 0x0000025986BA9C10>, <great_tables._gt_data.FormatInfo object at 0x0000025986BA9700>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=F

In [17]:
def split_table(models, pair_fe):
    t = pf.etable(models, type="df").drop("fe", level=0)
    t.loc[("sample", "Parent"), :]         = ["Italian", "Foreign"] * 3
    t.loc[("fe", "Firm x year FE"), :]     = ["Yes"] * 6
    t.loc[("fe", "Subclass x year FE"), :] = ["Yes"] * 6
    t.loc[("fe", "Firm-subclass FE"), :]   = [pair_fe] * 6
    return (t.reindex(["coef", "sample", "fe", "stats"], level=0)
             .replace({r"\n": " "}, regex=True))

st1 = split_table(s1, "No")
st2 = split_table(s2, "Yes")
st1

patented                                    \
                                       (1)              (2)              (3)   
coef   net_stock           0.000   (0.000)  0.000   (0.000)                    
       log_net                                               0.018   (0.010)   
       net_any                                                                 
sample Parent                      Italian          Foreign          Italian   
fe     Firm x year FE                  Yes              Yes              Yes   
       Subclass x year FE              Yes              Yes              Yes   
       Firm-subclass FE                 No               No               No   
stats  Observations                 16,510            6,020           16,510   
       R²                            0.382            0.458            0.384   

                                                                \
                                       (4)                 (5)   
coef   net_stock                                                 
       log_net             0.005   (0.015)                       
       net_any                              0.055***   (0.015)   
sample Parent                      Foreign             Italian   
fe     Firm x year FE                  Yes                 Yes   
       Subclass x year FE              Yes                 Yes   
       Firm-subclass FE                 No                  No   
stats  Observations                  6,020              16,510   
       R²                            0.458               0.386   

                                            
                                       (6)  
coef   net_stock                            
       log_net                              
       net_any             0.046   (0.053)  
sample Parent                      Foreign  
fe     Firm x year FE                  Yes  
       Subclass x year FE              Yes  
       Firm-subclass FE                 No  
stats  Observations                  6,020  
       R²                             0.46

In [18]:
st2

patented                    \
                                        (1)               (2)   
coef   net_stock           -0.000   (0.000)  -0.000   (0.001)   
       log_net                                                  
       net_any                                                  
sample Parent                       Italian           Foreign   
fe     Firm x year FE                   Yes               Yes   
       Subclass x year FE               Yes               Yes   
       Firm-subclass FE                 Yes               Yes   
stats  Observations                  16,510             6,020   
       R²                             0.611             0.675   

                                                              \
                                        (3)              (4)   
coef   net_stock                                               
       log_net             -0.032   (0.031)  0.030   (0.042)   
       net_any                                                 
sample Parent                       Italian          Foreign   
fe     Firm x year FE                   Yes              Yes   
       Subclass x year FE               Yes              Yes   
       Firm-subclass FE                 Yes              Yes   
stats  Observations                  16,510            6,020   
       R²                             0.611            0.676   

                                                               
                                        (5)               (6)  
coef   net_stock                                               
       log_net                                                 
       net_any             -0.025   (0.032)  -0.005   (0.050)  
sample Parent                       Italian           Foreign  
fe     Firm x year FE                   Yes               Yes  
       Subclass x year FE               Yes               Yes  
       Firm-subclass FE                 Yes               Yes  
stats  Observations                  16,510             6,020  
       R²                             0.611             0.675

In [19]:
for t, cap in [(st1, "Firm x year + subclass x year FE"),
               (st2, "Adding firm-subclass FE")]:
    o = t.copy()
    o.index = o.index.droplevel(0)
    o = o.rename(index={X: "Network stock", "log_net": "log(1 + stock)",
                        "net_any": "Network present", "R²": "$R^2$"})
    print(o.to_latex(escape=False, column_format="l" + "c" * 6,
                     caption=f"Network exposure by parent nationality — {cap}",
                     label="tab:split"))
    print()

\begin{table}
\caption{Network exposure by parent nationality — Firm x year + subclass x year FE}
\label{tab:split}
\begin{tabular}{lcccccc}
\toprule
 & \multicolumn{6}{r}{patented} \\
 & (1) & (2) & (3) & (4) & (5) & (6) \\
\midrule
Network stock & 0.000   (0.000) & 0.000   (0.000) &  &  &  &  \\
log(1 + stock) &  &  & 0.018   (0.010) & 0.005   (0.015) &  &  \\
Network present &  &  &  &  & 0.055***   (0.015) & 0.046   (0.053) \\
Parent & Italian & Foreign & Italian & Foreign & Italian & Foreign \\
Firm x year FE & Yes & Yes & Yes & Yes & Yes & Yes \\
Subclass x year FE & Yes & Yes & Yes & Yes & Yes & Yes \\
Firm-subclass FE & No & No & No & No & No & No \\
Observations & 16,510 & 6,020 & 16,510 & 6,020 & 16,510 & 6,020 \\
$R^2$ & 0.382 & 0.458 & 0.384 & 0.458 & 0.386 & 0.46 \\
\bottomrule
\end{tabular}
\end{table}


\begin{table}
\caption{Network exposure by parent nationality — Adding firm-subclass FE}
\label{tab:split}
\begin{tabular}{lcccccc}
\toprule
 & \multicolumn{6}{r}{patente

The pooled pattern holds in both subsamples. Cross-sectionally, `net_any` gives 0.055\*\*\*
(0.015) for Italian parents and 0.046 (0.053) for foreign — near-identical magnitudes, but
the foreign standard error is more than three times larger on 6,020 observations, so the
difference is precision, not effect size. `log_net` is weaker in both (0.018 and 0.005,
neither significant), and the levels form is uninformative throughout.

Adding the firm-subclass FE removes the effect in both subsamples: every coefficient falls
to zero or below, with standard errors larger than the point estimates. The opposite signs
on `log_net` (−0.032 Italian, +0.030 foreign) are noise at these standard errors, not a
heterogeneous effect.

So the one place multinational knowledge transfer should be most visible behaves exactly
like purely Italian groups — no distinct MNE channel on either margin — and the
cross-sectional/within-pair collapse is not driven by one side of the split.

### Inference check

The wild bootstrap is not feasible at this FE dimension: `wildboottest` one-hot encodes
fixed effects, and `firm_year + sub_year + pair` gives ~13,000 dummy columns. It is dropped
in notebook 4 as well, so both notebooks check inference the same way rather than holding
one specification to a different standard.

Instead, compare CRV1 at firm level against heteroskedasticity-robust and parent-level
clustering, across all three regressor forms. Parent level is the substantively motivated
one — co-affiliates' exposure is built from each other's patents, so sister firms are not
independent draws — and this settles the question left open in notebook 4. With 516 firm
clusters, agreement across the three estimators is sufficient reassurance.

In [20]:
inf5a = []
for v in [X, "log_net", "net_any"]:
    inf5a.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp, vcov={"CRV1": "firm_id"}))
    inf5a.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp, vcov="hetero"))
    inf5a.append(pf.feols(f"{Y} ~ {v} | {fe1}", data=rsp, vcov={"CRV1": PARENT}))

it5a = pf.etable(inf5a, type="df").drop("fe", level=0)

it5a.loc[("spec", "Standard errors"), :]  = ["Firm", "Robust", "Parent group"] * 3
it5a.loc[("fe", "Firm x year FE"), :]     = ["Yes"] * 9
it5a.loc[("fe", "Subclass x year FE"), :] = ["Yes"] * 9
it5a.loc[("fe", "Firm-subclass FE"), :]   = ["No"] * 9

it5a = it5a.reindex(["coef", "spec", "fe", "stats"], level=0)
it5a = it5a.replace({r"\n": " "}, regex=True)
it5a

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\

patented                                      \
                                      (1)                (2)              (3)   
coef  net_stock           0.000   (0.000)  0.000**   (0.000)  0.000   (0.000)   
      log_net                                                                   
      net_any                                                                   
spec  Standard errors                Firm             Robust     Parent group   
fe    Firm x year FE                  Yes                Yes              Yes   
      Subclass x year FE              Yes                Yes              Yes   
      Firm-subclass FE                 No                 No               No   
stats Observations                 24,320             24,320           24,320   
      R²                            0.334              0.334            0.334   

                                                                 \
                                        (4)                 (5)   
coef  net_stock                                                   
      log_net             0.025**   (0.009)  0.025***   (0.003)   
      net_any                                                     
spec  Standard errors                  Firm              Robust   
fe    Firm x year FE                    Yes                 Yes   
      Subclass x year FE                Yes                 Yes   
      Firm-subclass FE                   No                  No   
stats Observations                   24,320              24,320   
      R²                              0.337               0.337   

                                                                 \
                                        (6)                 (7)   
coef  net_stock                                                   
      log_net             0.025**   (0.009)                       
      net_any                                0.061***   (0.015)   
spec  Standard errors          Parent group                Firm   
fe    Firm x year FE                    Yes                 Yes   
      Subclass x year FE                Yes                 Yes   
      Firm-subclass FE                   No                  No   
stats Observations                   24,320              24,320   
      R²                              0.337               0.339   

                                                                  
                                         (8)                 (9)  
coef  net_stock                                                   
      log_net                                                     
      net_any             0.061***   (0.006)  0.061***   (0.016)  
spec  Standard errors                 Robust        Parent group  
fe    Firm x year FE                     Yes                 Yes  
      Subclass x year FE                 Yes                 Yes  
      Firm-subclass FE                    No                  No  
stats Observations                    24,320              24,320  
      R²                               0.339               0.339

In [21]:
inf5 = []
for v in [X, "log_net", "net_any"]:
    inf5.append(pf.feols(f"{Y} ~ {v} | {fe2}", data=rsp, vcov={"CRV1": "firm_id"}))
    inf5.append(pf.feols(f"{Y} ~ {v} | {fe2}", data=rsp, vcov="hetero"))
    inf5.append(pf.feols(f"{Y} ~ {v} | {fe2}", data=rsp, vcov={"CRV1": PARENT}))

it5 = pf.etable(inf5, type="df").drop("fe", level=0)

it5.loc[("spec", "Standard errors"), :]  = ["Firm", "Robust", "Parent group"] * 3
it5.loc[("fe", "Firm x year FE"), :]     = ["Yes"] * 9
it5.loc[("fe", "Subclass x year FE"), :] = ["Yes"] * 9
it5.loc[("fe", "Firm-subclass FE"), :]   = ["Yes"] * 9

it5 = it5.reindex(["coef", "spec", "fe", "stats"], level=0)
it5 = it5.replace({r"\n": " "}, regex=True)
it5

c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\IMT_studies\Lib\site-packages\pyfixest\estimation\formula\model_matrix.py:151: UserWarning: 3120 singleton fixed effect(s) dropped from the model.
  warnings.warn(
c:\Users\Popov\anaconda3\envs\

patented                    \
                                       (1)               (2)   
coef  net_stock           -0.000   (0.000)  -0.000   (0.000)   
      log_net                                                  
      net_any                                                  
spec  Standard errors                 Firm            Robust   
fe    Firm x year FE                   Yes               Yes   
      Subclass x year FE               Yes               Yes   
      Firm-subclass FE                 Yes               Yes   
stats Observations                  24,320            24,320   
      R²                             0.598             0.598   

                                                              \
                                       (3)               (4)   
coef  net_stock           -0.000   (0.000)                     
      log_net                               -0.011   (0.024)   
      net_any                                                  
spec  Standard errors         Parent group              Firm   
fe    Firm x year FE                   Yes               Yes   
      Subclass x year FE               Yes               Yes   
      Firm-subclass FE                 Yes               Yes   
stats Observations                  24,320            24,320   
      R²                             0.598             0.598   

                                                              \
                                       (5)               (6)   
coef  net_stock                                                
      log_net             -0.011   (0.018)  -0.011   (0.025)   
      net_any                                                  
spec  Standard errors               Robust      Parent group   
fe    Firm x year FE                   Yes               Yes   
      Subclass x year FE               Yes               Yes   
      Firm-subclass FE                 Yes               Yes   
stats Observations                  24,320            24,320   
      R²                             0.598             0.598   

                                                                                
                                       (7)               (8)               (9)  
coef  net_stock                                                                 
      log_net                                                                   
      net_any             -0.014   (0.028)  -0.014   (0.024)  -0.014   (0.033)  
spec  Standard errors                 Firm            Robust      Parent group  
fe    Firm x year FE                   Yes               Yes               Yes  
      Subclass x year FE               Yes               Yes               Yes  
      Firm-subclass FE                 Yes               Yes               Yes  
stats Observations                  24,320            24,320            24,320  
      R²                             0.598             0.598             0.598

In [22]:
for t, lab, cap in [(it5a, "tab:inf_nb5_a", "Firm x year + subclass x year FE"),
                    (it5,  "tab:inf_nb5_b", "Adding firm-subclass FE")]:
    o = t.copy()
    o.index = o.index.droplevel(0)
    o = o.rename(index={X: "Network stock", "log_net": "log(1 + stock)",
                        "net_any": "Network present", "R²": "$R^2$"})
    print(o.to_latex(escape=False, column_format="l" + "c" * 9,
                     caption=f"Risk-set estimates under alternative standard errors — {cap}",
                     label=lab))
    print()

\begin{table}
\caption{Risk-set estimates under alternative standard errors — Firm x year + subclass x year FE}
\label{tab:inf_nb5_a}
\begin{tabular}{lccccccccc}
\toprule
 & \multicolumn{9}{r}{patented} \\
 & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) & (9) \\
\midrule
Network stock & 0.000   (0.000) & 0.000**   (0.000) & 0.000   (0.000) &  &  &  &  &  &  \\
log(1 + stock) &  &  &  & 0.025**   (0.009) & 0.025***   (0.003) & 0.025**   (0.009) &  &  &  \\
Network present &  &  &  &  &  &  & 0.061***   (0.015) & 0.061***   (0.006) & 0.061***   (0.016) \\
Standard errors & Firm & Robust & Parent group & Firm & Robust & Parent group & Firm & Robust & Parent group \\
Firm x year FE & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes \\
Subclass x year FE & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes & Yes \\
Firm-subclass FE & No & No & No & No & No & No & No & No & No \\
Observations & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 & 24,320 \\
$R^2$ & 0.334 & 0


### What the inference check shows

Neither conclusion depends on the variance estimator.

Under `firm_year + sub_year`, `net_any` is 0.061 and significant at 0.1% under all three:
0.015 (firm), 0.006 (robust), 0.016 (parent group). `log_net` likewise holds at 0.025
throughout. Parent-group clustering barely widens the interval relative to firm level, so
residual dependence within corporate groups is small once the fixed effects are in.

Adding the pair FE drives everything to zero under all three estimators: `net_any` −0.014
with SEs of 0.028, 0.024 and 0.033; `log_net` −0.011 with 0.024, 0.018 and 0.025. The
standard errors are roughly twice the point estimates in every column.

Note what this settles and what it does not. It rules out the possibility that either
result is an artifact of clustering choice. It says nothing about sorting, which biases the
coefficient rather than the standard error — that question is answered by the
cross-sectional/within-pair contrast. And it does not address power: the within-pair
interval is wide because 163 pairs identify it, not because the variance was computed badly.

The levels form remains unreliable: significant under robust SEs in the cross-sectional
specification but not under either clustering, and the reverse pattern appeared in notebook
4. Treated as mechanical throughout.

The 3,120 singleton fixed effects dropped by `pyfixest` are firm-years and subclass-years
with a single observation; they carry no identifying variation.

## Summary

The risk-set design delivered what notebook 4 could not on the cross-sectional margin, and
less than hoped on the time margin.

**What worked.** The pre-2015 risk set keeps 516 of 674 firms and 3,311 firm-subclass
pairs, and 64% of firm-years hold two or more subclasses, so firm × year FE leave usable
variation. The cross-sectional estimate is large, precise and robust: within a firm-year,
across the technologies a firm already works in, those where the Italian network holds a
stock are 6.1pp more likely to see a new family — against an outcome mean of 3.6%. It holds
at 0.055\*\*\* for Italian parents and 0.046 for foreign, and under all three variance
estimators.

**What it means.** That contrast is between two different technologies within the same
firm-year, so it cannot separate a knowledge flow from shared technological identity. Adding
the firm-subclass FE forces identification onto variation over time and the effect
disappears — the signature of sorting. The within-pair interval excludes 0.061, so sorting
accounts for the bulk of the cross-sectional estimate.

**What stays open.** The within-pair arm is thin — 390 moving pairs, 163 with the outcome
also moving — so it rejects the cross-sectional magnitude without ruling out a moderate
effect (up to about +0.041, still larger than the outcome mean). The time margin is bounded,
not measured. An event study around network entry would settle it, but only 127 pairs see
the network enter a subclass inside the window, which is too few to estimate leads and lags.

**Where this points.** Notebook 6's variants are now the substantive next step rather than
robustness. Coarsening to the 3-character IPC class puts more firms in each technology and
should widen both arms; long differences trade ten noisy annual observations for a cleaner
pre/post contrast. If either restores precision on the time dimension, the sorting question
becomes answerable rather than bounded — and that, not the instrument, is what notebook 7
depends on.

Also unchanged from notebook 4: three independent cuts — rare intra-group citation (nb 2),
the underpowered extensive margin (nb 4), and the collapse documented here — point the same
way, toward shared technological identity rather than knowledge flow.